# BBC News: Business Category Analysis
## Traditional NLP Pipeline: TF-IDF + LDA Sub-category Discovery


In [57]:
# Core libraries
import os
import re
import sys

# Data handling
import pandas as pd
import numpy as np

# NLP — Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# NLP — Topic Modelling
from sklearn.decomposition import LatentDirichletAllocation

# Display
from rich.console import Console
from rich.table import Table

# Add parent directory to path so we can access loader.py
sys.path.append('..')
from loader import load_data

## Load the Data for Business only

In [58]:
# Load all articles and labels
documents, labels = load_data('../data')

# Filter business articles only
business_docs = [documents[i] for i in range(len(documents)) if labels[i] == 'business']

print(f"Total articles in dataset: {len(documents)}")
print(f"Business articles: {len(business_docs)}")
print(f"\nSample business article:\n{business_docs[0][:300]}")

Total articles in dataset: 2225
Business articles: 510

Sample business article:
UK economy facing 'major risks'

The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said.

The group's quarterly survey of companies found exports had picked up in the last three months of 2004 to their best level


## Clean the Data


In [59]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Custom stopwords — domain noise specific to BBC News
custom_stopwords = set([
    # Generic noise
    'said', 'says', 'say', 'told', 'added', 'according',
    'mr', 'mrs', 'ms', 'dr', 'sir',
    'year', 'years', 'quarter', 'quarters', 'month', 'months',
    'new', 'also', 'one', 'two', 'three', 'four', 'five',
    'would', 'could', 'may', 'will', 'going', 'come',
    'time', 'week', 'ago', 'last', 'first', 'second',
    'third', 'fourth', 'latest', 'previous', 'next',
    'figure', 'figures', 'result', 'results', 'rose',
    'fell', 'rise', 'fall', 'expected', 'forecast',
    'based', 'currently', 'however', 'despite', 'while',
    'since', 'back', 'well', 'still', 'already', 'just',
    'half', 'full', 'end', 'ended', 'ending','bn', 'ebbers', 'yuganskneftegas', 'baikal',
    'fraud', 'charges', 'court', 'legal', 'lawsuit',
    'december', 'january', 'february', 'march', 'april',
    'may', 'june', 'july', 'august', 'september',
    'october', 'november', 'gm', 'fiat', 'bmw',
    'mitsubishi', 'mercedes', 'motors', 'car', 'cars',
    'vehicles', 'lse', 'boerse', 'deutsche', 'euronext',
    'auction', 'argentina', 'dam', 'wmc', 'baikal',
    'rupees', 'lisbon', 'jobless'

    # Story-specific company names
    'yukos', 'yugansk', 'rosneft', 'gazprom', 'menatep',
    'worldcom', 'enron', 'healthsouth', 'parmalat',
    'timewarner', 'aol', 'pernod', 'domecq', 'seagram',
    'diageo', 'santander', 'qantas',

    # People names
    'greenspan', 'parsons', 'eddington', 'takenaka',
    'dixon', 'scrushy', 'khodorkovsky', 'botin',

    # Generic business words that appear everywhere
    'firm', 'company', 'companies', 'business', 'group',
    'market', 'markets', 'shares', 'share', 'profit',
    'profits', 'revenue', 'revenues', 'growth', 'quarter',
    'billion', 'million', 'percent', 'figure'

        # Yukos story
    'yukos', 'russian', 'russia', 'russias', 'rosneft',
    'yugansk', 'gazprom', 'menatep', 'khodorkovsky',
    'bankruptcy', 'tax', 'unit', 'sale', 'gas',

    # Glazer/Manchester United story  
    'glazer', 'glazers', 'manchester', 'club', 'united',
    'proposal', 'man',

    # Other story-specific noise
    'london', 'london stock exchange'
    ])


# Combine with sklearn stopwords
all_stopwords = ENGLISH_STOP_WORDS.union(custom_stopwords)

def clean_text(text):
    # Lowercase everything
    text = text.lower()
    
    # Remove text inside square brackets
    text = re.sub(r'\[.*?\]', '', text)
    
    # Remove non-letter characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stopwords — now using expanded list
    tokens = text.split()
    tokens = [word for word in tokens if word not in all_stopwords]
    
    return ' '.join(tokens)

In [60]:
# Apply cleaning to business articles
cleaned_business = [clean_text(doc) for doc in business_docs]

print(f"Cleaning complete — {len(cleaned_business)} business articles cleaned")
print(f"\nBefore cleaning:\n{business_docs[0][:200]}")
print(f"\nAfter cleaning:\n{cleaned_business[0][:200]}")


Cleaning complete — 510 business articles cleaned

Before cleaning:
UK economy facing 'major risks'

The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said.

The group's quarterly 

After cleaning:
uk economy facing major risks uk manufacturing sector continue face challenges british chamber commerce bcc groups quarterly survey exports picked best levels came exchange rates cited major concern b


## Remove Duplicates

In [61]:
# Remove duplicates from business articles
seen = set()
unique_business = []

for doc in cleaned_business:
    if doc not in seen:
        seen.add(doc)
        unique_business.append(doc)

print(f"Before deduplication: {len(cleaned_business)} articles")
print(f"After deduplication: {len(unique_business)} articles")
print(f"Duplicates removed: {len(cleaned_business) - len(unique_business)}")

Before deduplication: 510 articles
After deduplication: 503 articles
Duplicates removed: 7


## TF-IDF Feature Extraction

In [62]:
# Initialize TF-IDF Vectorizer for business articles
tfidf = TfidfVectorizer(
    max_features=1000,
    max_df=0.95,
    min_df=2
)

# Fit and transform business articles only
business_tfidf = tfidf.fit_transform(unique_business)

print(f"TF-IDF Matrix Shape: {business_tfidf.shape}")
print(f"\nThis means:")
print(f"  {business_tfidf.shape[0]} business articles (rows)")
print(f"  {business_tfidf.shape[1]} features/words (columns)")

feature_names = tfidf.get_feature_names_out()
print(f"\nTop 20 words in business vocabulary:")
print(feature_names[:20])

TF-IDF Matrix Shape: (503, 1000)

This means:
  503 business articles (rows)
  1000 features/words (columns)

Top 20 words in business vocabulary:
['able' 'abroad' 'accepted' 'access' 'account' 'accounting' 'accounts'
 'accused' 'acquisition' 'act' 'action' 'activity' 'adding'
 'administration' 'admitted' 'advertising' 'affected' 'africa' 'african'
 'age']


## LDA Topic Modelling (Sub-category Discovery)

In [63]:
# Run LDA on business articles
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(business_tfidf)

print("Business Sub-categories Discovered:")
print("=" * 50)

feature_names = tfidf.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"\nTopic {topic_idx + 1}:")
    print(f"  Top words: {', '.join(top_words)}")

Business Sub-categories Discovered:

Topic 1:
  Top words: euros, financial, accounting, deal, telecoms, stake, mobile, sullivan, mci, executives

Topic 2:
  Top words: sri, lanka, disaster, cairn, card, indonesia, tsunami, damage, reconstruction, thailand

Topic 3:
  Top words: sales, economy, oil, bank, economic, yukos, government, prices, uk, dollar


## Evaluate LDA with Coherence Score

In [64]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Prepare tokenized documents for gensim
tokenized_docs = [doc.split() for doc in unique_business]

# Build gensim dictionary
gensim_dict = Dictionary(tokenized_docs)

# Get top 10 words per topic as lists
topics_words = []
for topic in lda.components_:
    top_indices = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_indices]
    topics_words.append(top_words)

# Calculate coherence score
coherence_model = CoherenceModel(
    topics=topics_words,
    texts=tokenized_docs,
    dictionary=gensim_dict,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print(f"LDA Coherence Score (c_v): {coherence_score:.4f}")

# Interpretation
if coherence_score > 0.7:
    print("Excellent — topics are very well defined")
elif coherence_score > 0.5:
    print("Good — topics are clear and meaningful")
elif coherence_score > 0.3:
    print("Moderate — topics are somewhat meaningful")
else:
    print("Poor — consider adjusting number of topics")


LDA Coherence Score (c_v): 0.4619
Moderate — topics are somewhat meaningful


## Optimal Topic Count Evaluation

In [65]:
# Test different topic counts to find optimal coherence
results = []

for n in [3, 4, 5, 6]:
    lda_test = LatentDirichletAllocation(n_components=n, random_state=42)
    lda_test.fit(business_tfidf)
    
    topics_words_test = []
    for topic in lda_test.components_:
        top_indices = topic.argsort()[:-11:-1]
        top_words = [feature_names[i] for i in top_indices]
        topics_words_test.append(top_words)
    
    coherence_test = CoherenceModel(
        topics=topics_words_test,
        texts=tokenized_docs,
        dictionary=gensim_dict,
        coherence='c_v'
    ).get_coherence()
    
    results.append((n, coherence_test))
    print(f"Topics: {n} — Coherence: {coherence_test:.4f}")

print(f"\nBest number of topics: {max(results, key=lambda x: x[1])[0]}")

Topics: 3 — Coherence: 0.4619
Topics: 4 — Coherence: 0.5973
Topics: 5 — Coherence: 0.4462
Topics: 6 — Coherence: 0.4260

Best number of topics: 4


 ## LDA Topic Modelling Results

In [71]:
final_lda = LatentDirichletAllocation(n_components=4, random_state=42)
final_lda.fit(business_tfidf)

print("BBC BUSINESS — SUB-CATEGORY DISCOVERY RESULTS")
print("=" * 55)

for topic_idx, topic in enumerate(final_lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"\nTopic {topic_idx + 1}:")
    print(f"Key words: {', '.join(top_words)}")

print("\n" + "=" * 55)
print(f"Total business articles analysed: {len(unique_business)}")
print(f"Optimal number of sub-categories: 4")
print(f"Best coherence score: 0.5973")

BBC BUSINESS — SUB-CATEGORY DISCOVERY RESULTS

Topic 1:
Key words: yukos, deal, firms, financial, stake, takeover, chief, shareholders, giant, bid

Topic 2:
Key words: sri, disaster, lanka, tsunami, reconstruction, indonesia, damage, thailand, rebuilding, tourism

Topic 3:
Key words: airline, airlines, air, drug, bid, offer, jet, airways, barclays, stock

Topic 4:
Key words: sales, economy, economic, prices, bank, oil, dollar, government, uk, rate

Total business articles analysed: 503
Optimal number of sub-categories: 4
Best coherence score: 0.5973


##  NMF Model Comparison
### Evaluating NMF vs LDA for Business Sub-category Discovery

In [72]:
from sklearn.decomposition import NMF

# Run NMF with same number of topics as our best LDA
nmf = NMF(n_components=5, random_state=42)
nmf.fit(business_tfidf)

print("NMF — Business Sub-categories")
print("=" * 55)

for topic_idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"\nTopic {topic_idx + 1}:")
    print(f"Key words: {', '.join(top_words)}")

NMF — Business Sub-categories

Topic 1:
Key words: dollar, deficit, trade, oil, budget, china, economic, countries, euro, eu

Topic 2:
Key words: yukos, oil, assets, case, mikhail, sold, protection, authorities, state, bank

Topic 3:
Key words: bid, offer, deal, stock, takeover, shareholders, financial, stake, exchange, board

Topic 4:
Key words: sales, retail, stores, euros, christmas, strong, store, executive, retailers, net

Topic 5:
Key words: economy, rates, bank, rate, economic, consumer, prices, spending, manufacturing, inflation


## NMF Model Comparison

In [73]:
# Score NMF topics
nmf_topics_words = []
for topic in nmf.components_:
    top_indices = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_indices]
    nmf_topics_words.append(top_words)

nmf_coherence = CoherenceModel(
    topics=nmf_topics_words,
    texts=tokenized_docs,
    dictionary=gensim_dict,
    coherence='c_v'
).get_coherence()

print(f"NMF Coherence Score: {nmf_coherence:.4f}")
print(f"LDA Coherence Score: 0.5973")
print(f"\nWinner: {'NMF' if nmf_coherence > 0.5973 else 'LDA'}")

NMF Coherence Score: 0.6441
LDA Coherence Score: 0.5973

Winner: NMF


## Final Results: NMF Selected as Best Model


In [74]:
# Final model — NMF outperformed LDA (0.6441 vs best LDA 0.5973)
print("BBC BUSINESS — FINAL SUB-CATEGORY DISCOVERY RESULTS")
print("=" * 55)
print("Model: NMF (Non-negative Matrix Factorization)")
print(f"Coherence Score: 0.6441 (Excellent)")
print("=" * 55)

nmf_subcategory_labels = {
    0: "Global Trade & Currency Policy",
    1: "Energy Sector & State Regulation",
    2: "Mergers & Acquisitions",
    3: "Retail & Consumer Markets",
    4: "Macroeconomics & Monetary Policy"
}

for topic_idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    label = nmf_subcategory_labels[topic_idx]
    print(f"\nSub-category {topic_idx + 1}: {label}")
    print(f"Key words: {', '.join(top_words)}")

print("\n" + "=" * 55)
print(f"Total business articles analysed: {len(unique_business)}")
print(f"Optimal number of sub-categories: 5")
print(f"Model selected: NMF over LDA based on coherence score")

BBC BUSINESS — FINAL SUB-CATEGORY DISCOVERY RESULTS
Model: NMF (Non-negative Matrix Factorization)
Coherence Score: 0.6441 (Excellent)

Sub-category 1: Global Trade & Currency Policy
Key words: dollar, deficit, trade, oil, budget, china, economic, countries, euro, eu

Sub-category 2: Energy Sector & State Regulation
Key words: yukos, oil, assets, case, mikhail, sold, protection, authorities, state, bank

Sub-category 3: Mergers & Acquisitions
Key words: bid, offer, deal, stock, takeover, shareholders, financial, stake, exchange, board

Sub-category 4: Retail & Consumer Markets
Key words: sales, retail, stores, euros, christmas, strong, store, executive, retailers, net

Sub-category 5: Macroeconomics & Monetary Policy
Key words: economy, rates, bank, rate, economic, consumer, prices, spending, manufacturing, inflation

Total business articles analysed: 503
Optimal number of sub-categories: 5
Model selected: NMF over LDA based on coherence score
